In [ ]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

# list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)

In [ ]:
# Global Styles
SOURCE_STYLES = {
    "Zone":     {"color": "#1f77b4", "dash": "solid",   "width": 2},    
    "Outside":  {"color": "#ff7f0e", "dash": "dash",    "width": 0.5},  
    "Supply":   {"color": "#2ca02c", "dash": "dash",    "width": 0.5},  
    "Other_1":  {"color": "#FFBE91", "dash": "solid",   "width": 2},    
    "Setpoint": {"color": "#CFEBFF", "dash": "dot",     "width": 0.5},  
}

def build_zone_subplots(df, zone_name, subplot_config, source_styles=SOURCE_STYLES, row_height=250):
    total_rows = len(subplot_config)
    
    fig = make_subplots(
        rows=total_rows,
        cols=1,
        shared_xaxes=False,
        subplot_titles=[panel["title"] for panel in subplot_config],
        specs=[[{"secondary_y": True}]] * total_rows
    )
    
    for row_idx, panel in enumerate(subplot_config, start=1):
        
        # --- 1. Add Data Traces ---
        for trace_info in panel["traces"]:
            col_name = trace_info["col"]
            source_type = trace_info["source"]
            
            if col_name in df.columns:
                base_style = source_styles.get(source_type, {"color": "black", "dash": "solid", "width": 1})
                line_color = trace_info.get("color", base_style["color"])
                line_dash = trace_info.get("dash", base_style["dash"])
                line_width = trace_info.get("width", base_style["width"])
                
                display_name = trace_info.get("name", col_name)
                is_secondary = trace_info.get("secondary_y", False)
                
                fig.add_trace(
                    go.Scatter(
                        x=df["Datetime"],
                        y=df[col_name],
                        name=display_name,
                        
                        # --- SAFE LEGEND GROUPING ---
                        legendgroup=str(row_idx),
                        legendgrouptitle_text=f"<b>{panel['title']}</b>",
                        # ----------------------------
                        
                        line=dict(
                            color=line_color,
                            dash=line_dash,
                            width=line_width
                        )
                    ),
                    row=row_idx,
                    col=1,
                    secondary_y=is_secondary
                )

        # --- 2. Draw Expected Range ---
        if "expected_range" in panel:
            ymin, ymax = panel["expected_range"]
            range_label = panel.get("expected_label", "Expected Range")
            range_color = panel.get("range_color", "rgba(46, 204, 113, 0.15)") 
            
            fig.add_hrect(
                y0=ymin, y1=ymax,
                fillcolor=range_color,
                line_width=0, layer="below",
                row=row_idx, col=1,
                secondary_y=False
            )
            
            first_valid_time = df["Datetime"].iloc[0] if not df.empty else None
            fig.add_trace(
                go.Scatter(
                    x=[first_valid_time], y=[None],
                    mode="markers",
                    marker=dict(size=10, color=range_color, symbol="square"),
                    name=f"{range_label} ({ymin}-{ymax})",
                    
                    # --- SAFE LEGEND GROUPING ---
                    legendgroup=str(row_idx),
                    showlegend=True
                    # ----------------------------
                ),
                row=row_idx, col=1,
                secondary_y=False
            )
                
        # --- 3. Set Primary Y-axis Title and Range ---
        primary_y_kwargs = {"title_text": panel.get("y_label", "")}
        if "y_range" in panel:
            primary_y_kwargs["range"] = panel["y_range"]
        fig.update_yaxes(**primary_y_kwargs, row=row_idx, col=1, secondary_y=False)
        
        # --- 4. Set Secondary Y-axis Title and Range ---
        if "secondary_y_label" in panel or "secondary_y_range" in panel:
            secondary_y_kwargs = {}
            if "secondary_y_label" in panel:
                secondary_y_kwargs["title_text"] = panel["secondary_y_label"]
            if "secondary_y_range" in panel:
                secondary_y_kwargs["range"] = panel["secondary_y_range"]
            fig.update_yaxes(**secondary_y_kwargs, row=row_idx, col=1, secondary_y=True)
            
    # 5. Global Layout
    fig.update_layout(
        height=max(400, row_height * total_rows),
        title_text=f"{zone_name} Dashboard",
        hovermode="x unified",
        
        # --- Single, cleanly grouped legend on the right ---
        showlegend=True,
        legend=dict(
            groupclick="toggleitem", # Allows hiding individual lines instead of the whole group
            tracegroupgap=15         # Adds nice spacing between subplot groups
        ),
        margin=dict(r=150)
    )
    
    return fig

# EKF MONITOR

In [ ]:

config = [
    {
        "title": f"Temperature",
        "y_label": "Temperature (°C)",
        "y_range": [5, 35],
        "expected_range": [20, 22], 
        "range_color": "rgba(52, 152, 219, 0.05)",
        "expected_label": "Comfort Zone",
        "traces": [
            {"col": f"{zone}_Temp_C", "source": "Zone",    "name": "Zone Temp"},
            {"col": "Out_Temp_C",     "source": "Outside", "name": "Outdoor Temp"},
            {"col": "Fan_Out_Temp_C", "source": "Supply",  "name": "Supply Temp"},
        ]
    },
    {
        "title": f"Humidity Ratio",
        "y_label": "Humidity (kg/kg)",
        "expected_range": [0, 0.012],
        "range_color": "rgba(52, 152, 219, 0.05)",
        "traces": [
            {"col": f"{zone}_W_kg_kg", "source": "Zone",    "name": "Zone W"},
            {"col": "Out_W_kg_kg",     "source": "Outside", "name": "Outdoor W"},
            {"col": "Fan_Out_W_kg_kg", "source": "Supply",  "name": "Supply W"},
        ]
    },
    {
        "title": f"Relative Humidity",
        "y_label": "Relative Humidity (%)",
        "y_range": [0, 100],
        "expected_range": [30, 60],
        "range_color": "rgba(52, 152, 219, 0.05)",
        "expected_label": "Target RH Band",
        "traces": [
            {"col": f"{zone}_RH_pct", "source": "Zone",    "name": "Zone RH"},
            {"col": "Out_RH_pct",     "source": "Outside", "name": "Outdoor RH"},
            {"col": "Fan_Out_RH_pct", "source": "Supply",  "name": "Supply RH"},
        ]
    },
    {
        "title": f"CO2 & Occupancy",
        "y_label": "CO2 (ppm)",
        "secondary_y_label": "Occupants",  
        # "secondary_y_range": [0, 50],
        "y_range": [0, 1500],
        "expected_range": [0, 1000],
        "expected_label": "Acceptable CO2",
        "range_color": "rgba(52, 152, 219, 0.05)",
        "traces": [
            {"col": f"{zone}_CO2_ppm", "source": "Zone",    "name": "Zone CO2"},
            {"col": "Out_CO2_ppm",     "source": "Outside", "name": "Outdoor CO2"},
            {"col": "Fan_Out_CO2_ppm", "source": "Supply",  "name": "Supply CO2"},
            {"col": f"{zone}_Occupants", "source": "Setpoint",  "name": "No of Occupancy", "secondary_y": True}, 
        ]
    },
    {
        "title": f"Equipment Load & Occupancy",
        "y_label": "Load (W)",
        "secondary_y_label": "Occupants",  
        "traces": [
            {"col": f"{zone}_EquipLoad_W",   "source": "Zone",  "name": "Equip Load (W)"}, 
            {"col": f"{zone}_Occupants", "source": "Other_1",  "name": "No of Occupancy", "secondary_y": True}, 
        ]
    },

    {
        "title": f"VAV Flow & Equipment Status",
        "y_label": "Mass Flow (kg/s)", 
        "secondary_y_label": "Power (W) / Temp (°C)",
        "traces": [
            {"col": f"{zone}_VAV_Flow_kg_s", "source": "Zone",     "name": "VAV Flow"},
            {"col": f"{zone}_Flow_SP_kg_s",  "source": "Setpoint", "name": "Flow Setpoint"},
            # {"col": f"{zone}_Reheater_W",    "source": "Supply",   "name": "Reheater (W)",         "secondary_y": True, "color": "#e377c2", "dash": "solid"}, 
            # {"col": f"{zone}_Reheat_SP_C",   "source": "Setpoint", "name": "Reheat Setpoint (°C)", "secondary_y": True, "color": "#9467bd"}  
        ]
    },
]

# Loop from 1 to 5 (range stops before 6)
for i in range(1, 6):
    zone = f"SPACE{i}-1"
    
    fig = build_zone_subplots(df, zone, config)
    fig.show()
